# Phase 5 - Layer E: Candidate-Slot Generation and Weights

This notebook is the bridge from the realistic data pipeline back into the original optimization architecture.

Layer C produces raw physical visibility windows. Layer D enriches those windows with operational assumptions. Layer E explicitly performs the original discretization idea: each enriched visibility window can produce multiple fixed-length candidate slots.

Priority, urgency, normalized metrics, and final weights are assigned at the parent visibility-window level and then inherited by all child candidate slots. This keeps the operational value attached to the communication opportunity, while timing alternatives matter through feasibility and conflicts downstream.

The final optimization units are still candidate slots, preserving compatibility with the reduction, conflict-graph, MWIS, and QUBO workflow.

## Configuration

Layer E reads the Layer D output from `data/dataset/starlink/intermediate` and writes the final datasets (the full weighted candidate-slot table and the small, medium, and dense instances) directly to `data/dataset/starlink`. `MIN_CONTACT_DURATION_MINUTES` is the minimum useful scheduled contact duration, not the physical visibility duration. `STEP_SIZE_MINUTES` controls how densely candidate alternatives are generated inside each enriched visibility window.

For this first implementation, every generated candidate slot has the same duration within the scenario. Longer visibility windows therefore create more alternatives, not longer contacts. The duration term in the weight formula is kept for compatibility and future extension, but it is less informative while contact duration is fixed.

In [8]:
from pathlib import Path
import hashlib
import importlib.util
import json
import subprocess
import sys

REQUIRED_PACKAGES = ["pandas"]
missing_packages = [
    package_name
    for package_name in REQUIRED_PACKAGES
    if importlib.util.find_spec(package_name) is None
]

if missing_packages:
    print(f"Installing missing packages into this notebook kernel: {missing_packages}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])
    print("Dependency installation finished. If imports still fail, restart the kernel and run the notebook again.")

import pandas as pd

# --------- loading configuration environment ----------
# load .json config file shared by the data wrangling notebooks
CONFIG_PATH = Path("data_config.json")
data_config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))

MIN_CONTACT_DURATION_MINUTES = data_config["scheduling_min_contact_duration_minutes"]
STEP_SIZE_MINUTES = data_config["scheduling_step_size_minutes"]
PRIORITY_MODE = data_config["scheduling_priority_mode"]
URGENCY_MODE = data_config["scheduling_urgency_mode"]
PRIORITY_RANDOM_SEED = data_config["scheduling_priority_random_seed"]

SMALL_INSTANCE_SIZE = data_config["scheduling_small_instance_size"]
MEDIUM_INSTANCE_SIZE = data_config["scheduling_medium_instance_size"]
DENSE_INSTANCE_SIZE = data_config["scheduling_dense_instance_size"]
DENSE_STATION_COUNT = data_config["scheduling_dense_station_count"]

PRIORITY_WEIGHT = data_config["scheduling_priority_weight"]
DURATION_WEIGHT = data_config["scheduling_duration_weight"]
URGENCY_WEIGHT = data_config["scheduling_urgency_weight"]
ELEVATION_WEIGHT = data_config["scheduling_elevation_weight"]

# --------- project directory setup ----------
# final datasets (consumed by the optimization pipeline) live in DATA_DIR,
# the Layer D input comes from the intermediate subfolder
DATA_DIR = Path("../../data/dataset/starlink")
DATA_DIR.mkdir(parents=True, exist_ok=True)
INTERMEDIATE_DIR = DATA_DIR / "intermediate"
OPERATIONAL_INPUT_CSV = INTERMEDIATE_DIR / "starlink_visibility_windows_operational.csv"
FULL_OUTPUT_CSV = DATA_DIR / "starlink_scheduling_slots_weighted.csv"
SMALL_OUTPUT_CSV = DATA_DIR / "starlink_scheduling_instance_small.csv"
MEDIUM_OUTPUT_CSV = DATA_DIR / "starlink_scheduling_instance_medium.csv"
DENSE_OUTPUT_CSV = DATA_DIR / "starlink_scheduling_instance_dense.csv"

MIN_CONTACT_DURATION_SECONDS = MIN_CONTACT_DURATION_MINUTES * 60
STEP_SIZE_SECONDS = STEP_SIZE_MINUTES * 60
if STEP_SIZE_SECONDS <= 0 or MIN_CONTACT_DURATION_SECONDS <= 0:
    raise ValueError("Contact duration and step size must be positive.")
if STEP_SIZE_SECONDS > MIN_CONTACT_DURATION_SECONDS:
    raise ValueError("STEP_SIZE_MINUTES should be less than or equal to MIN_CONTACT_DURATION_MINUTES.")
if min(PRIORITY_WEIGHT, DURATION_WEIGHT, URGENCY_WEIGHT, ELEVATION_WEIGHT) < 0:
    raise ValueError("Weight coefficients in data_config.json cannot be negative.")

print(f"Notebook Python executable: {sys.executable}")
print(f"Config path: {CONFIG_PATH.resolve()}")
print(f"Layer E data directory: {DATA_DIR.resolve()}")
print(f"Operational input: {OPERATIONAL_INPUT_CSV.resolve()}")
print(f"Minimum useful contact duration: {MIN_CONTACT_DURATION_MINUTES} minutes")
print(f"Discretization step size: {STEP_SIZE_MINUTES} minutes")
print(f"Full candidate-slot output: {FULL_OUTPUT_CSV.resolve()}")

Notebook Python executable: d:\University\PoliMi\courses\SWE2\SWE2-research-project\.venv\Scripts\python.exe
Config path: D:\University\PoliMi\courses\SWE2\SWE2-research-project\results\01_data_wrangling\data_config.json
Layer E data directory: D:\University\PoliMi\courses\SWE2\SWE2-research-project\data\dataset\starlink
Operational input: D:\University\PoliMi\courses\SWE2\SWE2-research-project\data\dataset\starlink\intermediate\starlink_visibility_windows_operational.csv
Minimum useful contact duration: 4 minutes
Discretization step size: 2 minutes
Full candidate-slot output: D:\University\PoliMi\courses\SWE2\SWE2-research-project\data\dataset\starlink\starlink_scheduling_slots_weighted.csv


## Load Enriched Visibility Windows
Starting from Layer E enriched visibility windows, we extract the candidate windows by selecting only those visibility windows that don't overlapp with any blaockout(this check was done in te Layer E notebook).

In [9]:
def load_enriched_visibility_windows(input_csv_path: Path) -> pd.DataFrame:
    """Load Layer D enriched visibility windows for candidate-slot generation.

    Parameters
    ----------
    input_csv_path:
        Path to the Layer D enriched visibility-window CSV.
    include_unavailable:
        When True, keep windows flagged as operationally unavailable.

    Returns
    -------
    pandas.DataFrame
        Enriched visibility windows with parsed UTC timestamps.
    """
    
    windows_df = pd.read_csv(input_csv_path)
    required_columns = [
        "window_id",
        "satellite_name",
        "norad_id",
        "station_id",
        "station_name",
        "start_time",
        "end_time",
        "duration_seconds",
        "max_elevation_deg",
        "source_epoch",
        "antenna_type",
        "station_capacity",
        "reconfiguration_time_seconds",
        "is_operationally_available",
    ]
    missing_columns = [column for column in required_columns if column not in windows_df.columns]
    if missing_columns:
        raise ValueError(f"Missing required Layer D columns: {missing_columns}")

    # switching to UTC time format
    windows_df["start_time"] = pd.to_datetime(windows_df["start_time"], utc=True)
    windows_df["end_time"] = pd.to_datetime(windows_df["end_time"], utc=True)

    windows_df = windows_df.loc[windows_df["is_operationally_available"]].copy()

    windows_df = windows_df.sort_values(["start_time", "station_id", "norad_id"]).reset_index(drop=True)
    return windows_df


enriched_windows = load_enriched_visibility_windows(OPERATIONAL_INPUT_CSV)

print(f"Enriched visibility windows loaded for Layer E: {len(enriched_windows)}")
print(f"Unique satellites: {enriched_windows['norad_id'].nunique()}")
print(f"Unique stations: {enriched_windows['station_id'].nunique()}")
print("\nDuration summary for enriched windows, seconds:")
print(enriched_windows["duration_seconds"].describe().to_string())
enriched_windows.head(10)

Enriched visibility windows loaded for Layer E: 73600
Unique satellites: 983
Unique stations: 150

Duration summary for enriched windows, seconds:
count    73600.000000
mean       194.995109
std         64.757385
min         60.000000
25%        180.000000
50%        180.000000
75%        240.000000
max        300.000000


,window_id,satellite_name,norad_id,station_id,station_name,start_time,end_time,duration_seconds,max_elevation_deg,source_epoch,antenna_type,station_capacity,reconfiguration_time_seconds,availability_start,availability_end,blackout_window_ids,overlaps_blackout,is_operationally_available,operational_constraint_source
0,VIS_000001,STARLINK-1552,46076,GATEWAY_0001,"Adelanto, CA",2026-06-18 00:00:00+00:00,2026-06-18 00:04:00+00:00,240,48.621,2026-06-17T21:51:22Z,dish,1,180,2026-06-18T00:00:00+00:00,2026-06-18T04:00:00+00:00,NaN,False,True,deterministic synthetic Layer D assumptions
1,VIS_000002,STARLINK-2495,48325,GATEWAY_0001,"Adelanto, CA",2026-06-18 00:00:00+00:00,2026-06-18 00:03:00+00:00,180,50.028,2026-06-17T21:50:51Z,dish,1,180,2026-06-18T00:00:00+00:00,2026-06-18T04:00:00+00:00,NaN,False,True,deterministic synthetic Layer D assumptions
2,VIS_000003,STARLINK-2738,48668,GATEWAY_0001,"Adelanto, CA",2026-06-18 00:00:00+00:00,2026-06-18 00:04:00+00:00,240,66.572,2026-06-17T21:52:22Z,dish,1,180,2026-06-18T00:00:00+00:00,2026-06-18T04:00:00+00:00,NaN,False,True,deterministic synthetic Layer D assumptions
3,VIS_000004,STARLINK-3159,49415,GATEWAY_0001,"Adelanto, CA",2026-06-18 00:00:00+00:00,2026-06-18 00:02:00+00:00,120,32.655,2026-06-18T06:11:24Z,dish,1,180,2026-06-18T00:00:00+00:00,2026-06-18T04:00:00+00:00,NaN,False,True,deterministic synthetic Layer D assumptions
4,VIS_000005,STARLINK-3117,49435,GATEWAY_0001,"Adelanto, CA",2026-06-18 00:00:00+00:00,2026-06-18 00:02:00+00:00,120,50.149,2026-06-18T06:08:48Z,dish,1,180,2026-06-18T00:00:00+00:00,2026-06-18T04:00:00+00:00,NaN,False,True,deterministic synthetic Layer D assumptions
5,VIS_000006,STARLINK-2356,47896,GATEWAY_0002,"Aerzen, Germany",2026-06-18 00:00:00+00:00,2026-06-18 00:03:00+00:00,180,79.033,2026-06-17T23:34:49Z,dish,1,180,2026-06-18T00:00:00+00:00,2026-06-18T04:00:00+00:00,NaN,False,True,deterministic synthetic Layer D assumptions
6,VIS_000007,STARLINK-3297,50159,GATEWAY_0002,"Aerzen, Germany",2026-06-18 00:00:00+00:00,2026-06-18 00:05:00+00:00,300,88.099,2026-06-17T07:41:29Z,dish,1,180,2026-06-18T00:00:00+00:00,2026-06-18T04:00:00+00:00,NaN,False,True,deterministic synthetic Layer D assumptions
7,VIS_000008,STARLINK-3289,50173,GATEWAY_0002,"Aerzen, Germany",2026-06-18 00:00:00+00:00,2026-06-18 00:02:00+00:00,120,38.942,2026-06-18T02:40:02Z,dish,1,180,2026-06-18T00:00:00+00:00,2026-06-18T04:00:00+00:00,NaN,False,True,deterministic synthetic Layer D assumptions
8,VIS_000009,STARLINK-3283,50179,GATEWAY_0002,"Aerzen, Germany",2026-06-18 00:00:00+00:00,2026-06-18 00:01:00+00:00,60,28.556,2026-06-18T02:40:46Z,dish,1,180,2026-06-18T00:00:00+00:00,2026-06-18T04:00:00+00:00,NaN,False,True,deterministic synthetic Layer D assumptions
9,VIS_000010,STARLINK-3267,50197,GATEWAY_0002,"Aerzen, Germany",2026-06-18 00:00:00+00:00,2026-06-18 00:03:00+00:00,180,44.213,2026-06-18T02:42:05Z,dish,1,180,2026-06-18T00:00:00+00:00,2026-06-18T04:00:00+00:00,NaN,False,True,deterministic synthetic Layer D assumptions


## E1 - Generate Candidate Slots

Each candidate window is discretized into all fixed-length candidate slots that fit completely inside the parent window. If a parent window is shorter than the minimum useful contact duration, it generates no candidate slots.

In [10]:
def generate_candidate_slots_for_window(
    window: pd.Series,
    min_contact_seconds: int,
    step_seconds: int,
) -> list[dict]:
    """Generate fixed-length candidate slots inside one enriched visibility window.

    Parameters
    ----------
    window:
        One enriched visibility-window record from Layer D.
    min_contact_seconds:
        Fixed minimum useful scheduled contact duration.
    step_seconds:
        Discretization step between candidate start times.

    Returns
    -------
    list[dict]
        Candidate slot records that preserve traceability to the parent window.
    """
    window_start = window["start_time"]
    window_end = window["end_time"]
    window_duration_seconds = int((window_end - window_start).total_seconds())
    if window_duration_seconds < min_contact_seconds:
        return []

    candidate_records = []
    latest_start = window_end - pd.Timedelta(seconds=min_contact_seconds)
    current_start = window_start
    local_index = 1

    while current_start <= latest_start:
        current_end = current_start + pd.Timedelta(seconds=min_contact_seconds)
        antenna_id = f"ANT_{window['station_id']}"
        candidate_records.append(
            {
                "parent_window_id": window["window_id"],
                "source_interval_id": window["window_id"],
                "satellite_name": window["satellite_name"],
                "satellite_id": str(window["norad_id"]),
                "norad_id": int(window["norad_id"]),
                "antenna_id": antenna_id,
                "station_id": window["station_id"],
                "station_name": window["station_name"],
                "start_time": current_start.isoformat(),
                "end_time": current_end.isoformat(),
                "start_time_hours": (current_start - SCENARIO_START).total_seconds() / 3600,
                "end_time_hours": (current_end - SCENARIO_START).total_seconds() / 3600,
                "duration_seconds": min_contact_seconds,
                "duration_hours": min_contact_seconds / 3600,
                "parent_visibility_start": window_start.isoformat(),
                "parent_visibility_end": window_end.isoformat(),
                "parent_visibility_duration_seconds": window_duration_seconds,
                "candidate_index_in_parent": local_index,
                "max_elevation_deg": window["max_elevation_deg"],
                "source_epoch": window["source_epoch"],
                "antenna_type": window["antenna_type"],
                "station_capacity": window["station_capacity"],
                "reconfiguration_time_seconds": window["reconfiguration_time_seconds"],
                "reconfiguration_time_hours": window["reconfiguration_time_seconds"] / 3600,
                "is_operationally_available": window["is_operationally_available"],
            }
        )
        local_index += 1
        current_start += pd.Timedelta(seconds=step_seconds)

    return candidate_records


def generate_candidate_slots(
    windows_df: pd.DataFrame,
    min_contact_seconds: int,
    step_seconds: int,
) -> pd.DataFrame:
    """Generate candidate slots from all enriched visibility windows.

    Parameters
    ----------
    windows_df:
        Layer D enriched visibility-window table.
    min_contact_seconds:
        Fixed scheduled contact duration for each candidate slot.
    step_seconds:
        Discretization step between candidate start times.

    Returns
    -------
    pandas.DataFrame
        Candidate-slot table before priority, urgency, and weight assignment.
    """
    records = []
    for _, window in windows_df.iterrows():
        records.extend(generate_candidate_slots_for_window(window, min_contact_seconds, step_seconds))

    if not records:
        raise ValueError("No candidate slots were generated. Try lowering MIN_CONTACT_DURATION_MINUTES.")

    slots_df = pd.DataFrame.from_records(records)
    slots_df = slots_df.sort_values(["start_time", "station_id", "norad_id", "candidate_index_in_parent"]).reset_index(drop=True)
    slots_df.insert(0, "slot_id", [f"SLOT_{index:06d}" for index in range(1, len(slots_df) + 1)])
    return slots_df


SCENARIO_START = enriched_windows["start_time"].min().floor("min")
SCENARIO_END = enriched_windows["end_time"].max().ceil("min")
candidate_slots = generate_candidate_slots(enriched_windows, MIN_CONTACT_DURATION_SECONDS, STEP_SIZE_SECONDS)

parent_slot_counts = candidate_slots.groupby("parent_window_id").size()
print(f"Candidate slots generated: {len(candidate_slots)}")
print(f"Parent windows with at least one candidate slot: {parent_slot_counts.size}")
print(f"Parent windows with no candidate slots: {len(enriched_windows) - parent_slot_counts.size}")
print(f"Average candidate slots per generating parent window: {parent_slot_counts.mean():.2f}")

candidate_slots.head(10)

Candidate slots generated: 34751
Parent windows with at least one candidate slot: 34751
Parent windows with no candidate slots: 38849
Average candidate slots per generating parent window: 1.00


,slot_id,parent_window_id,source_interval_id,satellite_name,satellite_id,norad_id,antenna_id,station_id,station_name,start_time,...,parent_visibility_end,parent_visibility_duration_seconds,candidate_index_in_parent,max_elevation_deg,source_epoch,antenna_type,station_capacity,reconfiguration_time_seconds,reconfiguration_time_hours,is_operationally_available
0,SLOT_000001,VIS_000001,VIS_000001,STARLINK-1552,46076,46076,ANT_GATEWAY_0001,GATEWAY_0001,"Adelanto, CA",2026-06-18T00:00:00+00:00,...,2026-06-18T00:04:00+00:00,240,1,48.621,2026-06-17T21:51:22Z,dish,1,180,0.05,True
1,SLOT_000002,VIS_000003,VIS_000003,STARLINK-2738,48668,48668,ANT_GATEWAY_0001,GATEWAY_0001,"Adelanto, CA",2026-06-18T00:00:00+00:00,...,2026-06-18T00:04:00+00:00,240,1,66.572,2026-06-17T21:52:22Z,dish,1,180,0.05,True
2,SLOT_000003,VIS_000007,VIS_000007,STARLINK-3297,50159,50159,ANT_GATEWAY_0002,GATEWAY_0002,"Aerzen, Germany",2026-06-18T00:00:00+00:00,...,2026-06-18T00:05:00+00:00,300,1,88.099,2026-06-17T07:41:29Z,dish,1,180,0.05,True
3,SLOT_000004,VIS_000012,VIS_000012,STARLINK-3364,51152,51152,ANT_GATEWAY_0002,GATEWAY_0002,"Aerzen, Germany",2026-06-18T00:00:00+00:00,...,2026-06-18T00:05:00+00:00,300,1,54.613,2026-06-17T09:24:42Z,dish,1,180,0.05,True
4,SLOT_000005,VIS_000022,VIS_000022,STARLINK-2214,48566,48566,ANT_GATEWAY_0004,GATEWAY_0004,"Akron, OH",2026-06-18T00:00:00+00:00,...,2026-06-18T00:04:00+00:00,240,1,59.675,2026-06-18T01:02:19Z,dish,1,180,0.05,True
5,SLOT_000006,VIS_000024,VIS_000024,STARLINK-3046,49169,49169,ANT_GATEWAY_0004,GATEWAY_0004,"Akron, OH",2026-06-18T00:00:00+00:00,...,2026-06-18T00:04:00+00:00,240,1,73.793,2026-06-18T01:00:56Z,dish,1,180,0.05,True
6,SLOT_000007,VIS_000027,VIS_000027,STARLINK-3203,49761,49761,ANT_GATEWAY_0004,GATEWAY_0004,"Akron, OH",2026-06-18T00:00:00+00:00,...,2026-06-18T00:04:00+00:00,240,1,68.915,2026-06-18T01:04:03Z,dish,1,180,0.05,True
7,SLOT_000008,VIS_000030,VIS_000030,STARLINK-2015,47638,47638,ANT_GATEWAY_0005,GATEWAY_0005,"Alfouvar de Cima, Portugal",2026-06-18T00:00:00+00:00,...,2026-06-18T00:04:00+00:00,240,1,47.655,2026-06-17T09:45:03Z,dish,1,180,0.05,True
8,SLOT_000009,VIS_000038,VIS_000038,STARLINK-2151,48556,48556,ANT_GATEWAY_0007,GATEWAY_0007,"Anderson, SC",2026-06-18T00:00:00+00:00,...,2026-06-18T00:04:00+00:00,240,1,76.106,2026-06-18T01:00:06Z,dish,1,180,0.05,True
9,SLOT_000010,VIS_000039,VIS_000039,STARLINK-3088,49163,49163,ANT_GATEWAY_0007,GATEWAY_0007,"Anderson, SC",2026-06-18T00:00:00+00:00,...,2026-06-18T00:04:00+00:00,240,1,53.984,2026-06-18T01:00:00Z,dish,1,180,0.05,True


## E2 - Assign Parent Opportunity Metadata

Priority, urgency, normalized metrics, and weight are computed once per enriched visibility window. Every child candidate slot from the same parent inherits those values unchanged. This makes alternatives from the same visibility opportunity equally valuable.

In [11]:
def deterministic_integer(min_value: int, max_value: int, *parts: object) -> int:
    """Return a stable pseudo-random integer in a closed interval.

    Parameters
    ----------
    min_value:
        Smallest integer that can be returned.
    max_value:
        Largest integer that can be returned.
    *parts:
        Values included in the deterministic hash.

    Returns
    -------
    int
        Deterministic integer between `min_value` and `max_value`.
    """
    if min_value > max_value:
        raise ValueError("min_value cannot be greater than max_value.")

    text = "|".join(str(part) for part in parts)
    digest = hashlib.sha256(text.encode("utf-8")).hexdigest()
    span = max_value - min_value + 1
    return min_value + (int(digest[:12], 16) % span)


def assign_parent_priority(row: pd.Series) -> int:
    """Assign deterministic priority to one parent visibility window.

    Parameters
    ----------
    row:
        Enriched visibility-window row.

    Returns
    -------
    int
        Priority score from 1 to 5, where 5 is highest priority.
    """
    base_priority = deterministic_integer(1, 5, PRIORITY_RANDOM_SEED, row["norad_id"])
    if row["max_elevation_deg"] >= 60:
        base_priority += 1
    elif row["max_elevation_deg"] < 20:
        base_priority -= 1
    return min(5, max(1, base_priority))


def compute_parent_opportunity_metadata(parent_windows_df: pd.DataFrame) -> pd.DataFrame:
    """Compute priority, urgency, normalized metrics, and weight per parent window.

    Parameters
    ----------
    parent_windows_df:
        Enriched visibility windows that are eligible for candidate-slot generation.

    Returns
    -------
    pandas.DataFrame
        Parent-window metadata keyed by `parent_window_id` for child-slot inheritance.
    """
    parents = parent_windows_df.copy()
    horizon_seconds = max(1.0, (SCENARIO_END - SCENARIO_START).total_seconds())

    parents["priority"] = parents.apply(assign_parent_priority, axis=1)
    seconds_until_window_start = (parents["start_time"] - SCENARIO_START).dt.total_seconds().clip(lower=0)
    urgency_norm = (1.0 - (seconds_until_window_start / horizon_seconds)).clip(lower=0, upper=1)
    parents["urgency"] = (1 + (urgency_norm * 4)).round().astype(int).clip(lower=1, upper=5)

    max_visibility_duration = max(float(parents["duration_seconds"].max()), 1.0)
    parents["priority_norm"] = (parents["priority"] - 1) / 4
    parents["duration_norm"] = (parents["duration_seconds"] / max_visibility_duration).clip(lower=0, upper=1)
    parents["urgency_norm"] = (parents["urgency"] - 1) / 4
    parents["elevation_norm"] = (parents["max_elevation_deg"] / 90).clip(lower=0, upper=1)

    parents["weight"] = (
        100
        * (
            PRIORITY_WEIGHT * parents["priority_norm"]
            + DURATION_WEIGHT * parents["duration_norm"]
            + URGENCY_WEIGHT * parents["urgency_norm"]
            + ELEVATION_WEIGHT * parents["elevation_norm"]
        )
    ).round(3)

    return parents[
        [
            "window_id",
            "priority",
            "urgency",
            "priority_norm",
            "duration_norm",
            "urgency_norm",
            "elevation_norm",
            "weight",
        ]
    ].rename(columns={"window_id": "parent_window_id"})


def add_inherited_candidate_slot_weights(slots_df: pd.DataFrame, parent_windows_df: pd.DataFrame) -> pd.DataFrame:
    """Attach parent-level optimization metadata to every generated candidate slot.

    Parameters
    ----------
    slots_df:
        Candidate slots generated from enriched visibility windows.
    parent_windows_df:
        Enriched visibility windows used as candidate-slot parents.

    Returns
    -------
    pandas.DataFrame
        Candidate slots with inherited parent opportunity metadata.
    """
    parent_metadata = compute_parent_opportunity_metadata(parent_windows_df)
    slots = slots_df.merge(parent_metadata, on="parent_window_id", how="left")

    if slots["weight"].isna().any():
        raise ValueError("Some candidate slots could not inherit parent opportunity metadata.")

    slots["scenario_tag"] = "starlink_realistic_discretized_full"
    slots["priority_mode"] = PRIORITY_MODE
    slots["urgency_mode"] = URGENCY_MODE
    slots["weight_scope"] = "parent_visibility_window"
    slots["min_contact_duration_seconds"] = MIN_CONTACT_DURATION_SECONDS
    slots["discretization_step_seconds"] = STEP_SIZE_SECONDS
    return slots


weighted_slots = add_inherited_candidate_slot_weights(candidate_slots, enriched_windows)

same_parent_score_counts = weighted_slots.groupby("parent_window_id")[["priority", "urgency", "weight"]].nunique()
parents_with_varying_scores = int((same_parent_score_counts > 1).any(axis=1).sum())

print(f"Weighted candidate slots: {len(weighted_slots)}")
print(f"Parents with varying child priority/urgency/weight: {parents_with_varying_scores}")
print("Priority counts:")
print(weighted_slots["priority"].value_counts().sort_index().to_string())
print("\nUrgency counts:")
print(weighted_slots["urgency"].value_counts().sort_index().to_string())
weighted_slots.head(10)

Weighted candidate slots: 34751
Parents with varying child priority/urgency/weight: 0
Priority counts:
priority
1     3243
2     6854
3     6867
4     7089
5    10698

Urgency counts:
urgency
1    3223
2    8950
3    8620
4    9468
5    4490


,slot_id,parent_window_id,source_interval_id,satellite_name,satellite_id,norad_id,antenna_id,station_id,station_name,start_time,...,duration_norm,urgency_norm,elevation_norm,weight,scenario_tag,priority_mode,urgency_mode,weight_scope,min_contact_duration_seconds,discretization_step_seconds
0,SLOT_000001,VIS_000001,VIS_000001,STARLINK-1552,46076,46076,ANT_GATEWAY_0001,GATEWAY_0001,"Adelanto, CA",2026-06-18T00:00:00+00:00,...,0.8,1.0,0.540233,90.402,starlink_realistic_discretized_full,satellite_stable_with_pass_quality,earlier_slots_more_urgent,parent_visibility_window,240,120
1,SLOT_000002,VIS_000003,VIS_000003,STARLINK-2738,48668,48668,ANT_GATEWAY_0001,GATEWAY_0001,"Adelanto, CA",2026-06-18T00:00:00+00:00,...,0.8,1.0,0.739689,69.897,starlink_realistic_discretized_full,satellite_stable_with_pass_quality,earlier_slots_more_urgent,parent_visibility_window,240,120
2,SLOT_000003,VIS_000007,VIS_000007,STARLINK-3297,50159,50159,ANT_GATEWAY_0002,GATEWAY_0002,"Aerzen, Germany",2026-06-18T00:00:00+00:00,...,1.0,1.0,0.978878,77.289,starlink_realistic_discretized_full,satellite_stable_with_pass_quality,earlier_slots_more_urgent,parent_visibility_window,240,120
3,SLOT_000004,VIS_000012,VIS_000012,STARLINK-3364,51152,51152,ANT_GATEWAY_0002,GATEWAY_0002,"Aerzen, Germany",2026-06-18T00:00:00+00:00,...,1.0,1.0,0.606811,96.068,starlink_realistic_discretized_full,satellite_stable_with_pass_quality,earlier_slots_more_urgent,parent_visibility_window,240,120
4,SLOT_000005,VIS_000022,VIS_000022,STARLINK-2214,48566,48566,ANT_GATEWAY_0004,GATEWAY_0004,"Akron, OH",2026-06-18T00:00:00+00:00,...,0.8,1.0,0.663056,57.881,starlink_realistic_discretized_full,satellite_stable_with_pass_quality,earlier_slots_more_urgent,parent_visibility_window,240,120
5,SLOT_000006,VIS_000024,VIS_000024,STARLINK-3046,49169,49169,ANT_GATEWAY_0004,GATEWAY_0004,"Akron, OH",2026-06-18T00:00:00+00:00,...,0.8,1.0,0.819922,93.199,starlink_realistic_discretized_full,satellite_stable_with_pass_quality,earlier_slots_more_urgent,parent_visibility_window,240,120
6,SLOT_000007,VIS_000027,VIS_000027,STARLINK-3203,49761,49761,ANT_GATEWAY_0004,GATEWAY_0004,"Akron, OH",2026-06-18T00:00:00+00:00,...,0.8,1.0,0.765722,92.657,starlink_realistic_discretized_full,satellite_stable_with_pass_quality,earlier_slots_more_urgent,parent_visibility_window,240,120
7,SLOT_000008,VIS_000030,VIS_000030,STARLINK-2015,47638,47638,ANT_GATEWAY_0005,GATEWAY_0005,"Alfouvar de Cima, Portugal",2026-06-18T00:00:00+00:00,...,0.8,1.0,0.529500,79.045,starlink_realistic_discretized_full,satellite_stable_with_pass_quality,earlier_slots_more_urgent,parent_visibility_window,240,120
8,SLOT_000009,VIS_000038,VIS_000038,STARLINK-2151,48556,48556,ANT_GATEWAY_0007,GATEWAY_0007,"Anderson, SC",2026-06-18T00:00:00+00:00,...,0.8,1.0,0.845622,93.456,starlink_realistic_discretized_full,satellite_stable_with_pass_quality,earlier_slots_more_urgent,parent_visibility_window,240,120
9,SLOT_000010,VIS_000039,VIS_000039,STARLINK-3088,49163,49163,ANT_GATEWAY_0007,GATEWAY_0007,"Anderson, SC",2026-06-18T00:00:00+00:00,...,0.8,1.0,0.599822,57.248,starlink_realistic_discretized_full,satellite_stable_with_pass_quality,earlier_slots_more_urgent,parent_visibility_window,240,120


## Build Reviewable Scheduling Instances

The small and medium instances are chronological subsets of candidate slots. The dense instance focuses on the busiest station resources, increasing same-station overlap and reconfiguration pressure for later conflict-graph experiments.

In [12]:
def finalize_instance(slots_df: pd.DataFrame, scenario_tag: str, max_rows: int) -> pd.DataFrame:
    """Create a scheduling instance with local slot IDs and a scenario tag.

    Parameters
    ----------
    slots_df:
        Weighted candidate slots to include.
    scenario_tag:
        Scenario label saved on every row.
    max_rows:
        Maximum number of rows to retain.

    Returns
    -------
    pandas.DataFrame
        Instance table sorted by start time and weight.
    """
    instance = (
        slots_df.sort_values(["start_time", "station_id", "weight"], ascending=[True, True, False])
        .head(max_rows)
        .copy()
        .reset_index(drop=True)
    )
    instance["slot_id"] = [f"{scenario_tag.upper()}_{index:05d}" for index in range(1, len(instance) + 1)]
    instance["scenario_tag"] = scenario_tag
    return instance


def build_dense_conflict_instance(slots_df: pd.DataFrame, station_count: int, max_rows: int) -> pd.DataFrame:
    """Create a denser candidate-slot instance from the busiest station resources.

    Parameters
    ----------
    slots_df:
        Full weighted candidate-slot table.
    station_count:
        Number of high-traffic stations to include.
    max_rows:
        Maximum number of rows retained in the dense instance.

    Returns
    -------
    pandas.DataFrame
        Conflict-heavy instance biased toward shared station resources.
    """
    busiest_stations = (
        slots_df.groupby("station_id")
        .size()
        .sort_values(ascending=False)
        .head(station_count)
        .index
    )
    dense_candidates = slots_df.loc[slots_df["station_id"].isin(busiest_stations)].copy()
    dense_candidates = dense_candidates.sort_values(["station_id", "start_time", "weight"], ascending=[True, True, False])
    return finalize_instance(dense_candidates, "dense_station_conflict", max_rows)


small_instance = finalize_instance(weighted_slots, "small_discretized", SMALL_INSTANCE_SIZE)
medium_instance = finalize_instance(weighted_slots, "medium_discretized", MEDIUM_INSTANCE_SIZE)
dense_instance = build_dense_conflict_instance(weighted_slots, DENSE_STATION_COUNT, DENSE_INSTANCE_SIZE)

print(f"Small instance rows: {len(small_instance)}")
print(f"Medium instance rows: {len(medium_instance)}")
print(f"Dense instance rows: {len(dense_instance)}")
print("\nDense instance station counts:")
print(dense_instance["station_name"].value_counts().head(10).to_string())

Small instance rows: 100
Medium instance rows: 500
Dense instance rows: 700

Dense instance station counts:
station_name
Bellingham, WA    136
Colburn, ID       124
Hitterdal, MN     123
Conrad, MT        116
Brewster, WA      104
Kent, WA           97


## Sanity Check and Example

A quick check that the E1 discretization never places a candidate slot outside its parent visibility window, followed by an example: the parent window with the most child slots and the overlapping alternatives generated from it, all sharing the same inherited priority, urgency, and weight.

In [13]:
# sanity check on E1 discretization: every slot must lie inside its parent window
parent_bounds = enriched_windows.set_index("window_id")[["start_time", "end_time"]]
slot_parent_start = weighted_slots["parent_window_id"].map(parent_bounds["start_time"])
slot_parent_end = weighted_slots["parent_window_id"].map(parent_bounds["end_time"])
slot_start = pd.to_datetime(weighted_slots["start_time"], utc=True)
slot_end = pd.to_datetime(weighted_slots["end_time"], utc=True)
assert ((slot_start >= slot_parent_start) & (slot_end <= slot_parent_end)).all(), "Some slots fall outside their parent window."

# example: the parent window with the most child slots
example_parent_id = parent_slot_counts.idxmax()
print(enriched_windows.loc[enriched_windows["window_id"].eq(example_parent_id), ["window_id", "satellite_name", "station_name", "start_time", "end_time", "duration_seconds"]].to_string(index=False))
weighted_slots.loc[weighted_slots["parent_window_id"].eq(example_parent_id), ["slot_id", "start_time", "end_time", "priority", "urgency", "weight"]]

 window_id satellite_name station_name                start_time                  end_time  duration_seconds
VIS_000001  STARLINK-1552 Adelanto, CA 2026-06-18 00:00:00+00:00 2026-06-18 00:04:00+00:00               240


,slot_id,start_time,end_time,priority,urgency,weight
0,SLOT_000001,2026-06-18T00:00:00+00:00,2026-06-18T00:04:00+00:00,5,5,90.402


## Save Phase 5 Outputs

The full weighted table is the candidate-slot dataset produced by Layer E. The small, medium, and dense CSVs are reviewable optimization instances derived from those candidate slots.

In [14]:
weighted_slots.to_csv(FULL_OUTPUT_CSV, index=False)
small_instance.to_csv(SMALL_OUTPUT_CSV, index=False)
medium_instance.to_csv(MEDIUM_OUTPUT_CSV, index=False)
dense_instance.to_csv(DENSE_OUTPUT_CSV, index=False)

print(f"Saved full weighted candidate slots to: {FULL_OUTPUT_CSV.resolve()}")
print(f"Saved small instance to: {SMALL_OUTPUT_CSV.resolve()}")
print(f"Saved medium instance to: {MEDIUM_OUTPUT_CSV.resolve()}")
print(f"Saved dense instance to: {DENSE_OUTPUT_CSV.resolve()}")
print(f"Rows saved: full={len(weighted_slots)}, small={len(small_instance)}, medium={len(medium_instance)}, dense={len(dense_instance)}")

print(f"Satellites: {weighted_slots[["satellite_name"]].nunique()}")
print(f"Ground stations: {weighted_slots[["station_id"]].nunique()}")

Saved full weighted candidate slots to: D:\University\PoliMi\courses\SWE2\SWE2-research-project\data\dataset\starlink\starlink_scheduling_slots_weighted.csv
Saved small instance to: D:\University\PoliMi\courses\SWE2\SWE2-research-project\data\dataset\starlink\starlink_scheduling_instance_small.csv
Saved medium instance to: D:\University\PoliMi\courses\SWE2\SWE2-research-project\data\dataset\starlink\starlink_scheduling_instance_medium.csv
Saved dense instance to: D:\University\PoliMi\courses\SWE2\SWE2-research-project\data\dataset\starlink\starlink_scheduling_instance_dense.csv
Rows saved: full=34751, small=100, medium=500, dense=700
Satellites: satellite_name    836
dtype: int64
Ground stations: station_id    150
dtype: int64
